# IEEE 57-Bus DC-OPF with All Constraints
**Author:** Jewook Park  
**Date:** October 2025  
**Description:** DC-OPF with line switching, angle bounds (±30°), and capacity cuts for IEEE 57-bus system


In [19]:
import os
os.environ["GRB_LICENSE_FILE"] = "/Users/a/Desktop/VIP/sc-opf/API key/gurobi.lic"
import re, pandas as pd, numpy as np, gurobipy as gp, matplotlib.pyplot as plt, networkx as nx
from gurobipy import GRB
print("All libraries loaded!")


All libraries loaded!


In [20]:
def extract_matrix_block(lines, varname):
    in_block, matrix_lines = False, []
    for line in lines:
        if line.strip().startswith(f"{varname} = ["): in_block = True; continue
        if in_block:
            if line.strip().startswith("];"): break
            clean = re.sub(r'%.*', '', line).strip().rstrip(';')
            if clean: matrix_lines.append(clean)
    return np.array([[float(x) for x in line.split()] for line in matrix_lines])

with open('data/pglib_opf_case57_ieee.m', 'r') as f: lines = f.readlines()
bus_df = pd.DataFrame(extract_matrix_block(lines, 'mpc.bus'), columns=['bus_i','type','Pd','Qd','Gs','Bs','area','Vm','Va','baseKV','zone','Vmax','Vmin'])
gen_df = pd.DataFrame(extract_matrix_block(lines, 'mpc.gen'), columns=['bus','Pg','Qg','Qmax','Qmin','Vg','mBase','status','Pmax','Pmin'])
branch_df = pd.DataFrame(extract_matrix_block(lines, 'mpc.branch'), columns=['fbus','tbus','r','x','b','rateA','rateB','rateC','ratio','angle','status','angmin','angmax'])
gencost_df = pd.DataFrame(extract_matrix_block(lines, 'mpc.gencost'), columns=['model','startup','shutdown','n','c2','c1','c0'])
print(f"IEEE 57-bus: {len(bus_df)} buses, {len(branch_df)} branches, {len(gen_df)} generators")


IEEE 57-bus: 57 buses, 80 branches, 7 generators


In [21]:
def solve_dc_opf_full(bus_df, gen_df, branch_df, gencost_df, capacity_factor=0.9):
    """DC-OPF with ALL constraints: line switching + angle bounds + capacity cuts"""
    model = gp.Model("DC-OPF-Full-Case57")
    model.Params.OutputFlag = 0
    ref_bus = bus_df[bus_df['type'] == 3]['bus_i'].values[0]
    
    # Angle bounds
    MAX_ANGLE_RAD = 0.698  # ±40 degrees
    max_susceptance = (1.0 / branch_df['x']).max()
    max_capacity = branch_df['rateA'].max()
    M = max(2 * MAX_ANGLE_RAD * max_susceptance, max_capacity * 2)
    
    print(f"Angle bounds: ±{MAX_ANGLE_RAD * 180 / np.pi:.1f}° | Capacity factor: {capacity_factor} | Big-M: {M:.2f}")
    
    # Variables with angle bounds
    Pg = model.addVars(gen_df.index, lb=0, name="Pg")
    theta = model.addVars(bus_df['bus_i'], lb=-MAX_ANGLE_RAD, ub=MAX_ANGLE_RAD, name="theta")
    P_branch = model.addVars(branch_df.index, lb=-GRB.INFINITY, name="P_branch")
    z = model.addVars(branch_df.index, vtype=GRB.BINARY, name="z")
    
    # Objective
    obj = gp.QuadExpr()
    for idx, row in gencost_df.iterrows():
        obj += row['c2'] * Pg[idx]*Pg[idx] + row['c1'] * Pg[idx] + row['c0']
    model.setObjective(obj, GRB.MINIMIZE)
    
    # Constraints
    model.addConstr(theta[ref_bus] == 0)
    
    for idx, row in gen_df.iterrows():
        model.addConstr(Pg[idx] >= row['Pmin'])
        model.addConstr(Pg[idx] <= row['Pmax'])
    
    for idx, row in branch_df.iterrows():
        fbus, tbus, x = row['fbus'], row['tbus'], row['x']
        B = 1.0 / x
        model.addConstr(P_branch[idx] - B * (theta[fbus] - theta[tbus]) <= M * (1 - z[idx]))
        model.addConstr(P_branch[idx] - B * (theta[fbus] - theta[tbus]) >= -M * (1 - z[idx]))
        model.addConstr(P_branch[idx] <= M * z[idx])
        model.addConstr(P_branch[idx] >= -M * z[idx])
    
    # Capacity constraints with cut
    for idx, row in branch_df.iterrows():
        rateA = row['rateA'] * capacity_factor
        if row['rateA'] > 0:
            model.addConstr(P_branch[idx] <= rateA * z[idx])
            model.addConstr(P_branch[idx] >= -rateA * z[idx])
    
    for idx, row in bus_df.iterrows():
        bus, Pd = row['bus_i'], row['Pd']
        gen_at_bus = gen_df[gen_df['bus'] == bus].index.tolist()
        gen_P = gp.quicksum(Pg[g] for g in gen_at_bus) if gen_at_bus else 0
        branch_out = gp.quicksum(P_branch[br] for br in branch_df[branch_df['fbus'] == bus].index)
        branch_in = gp.quicksum(P_branch[br] for br in branch_df[branch_df['tbus'] == bus].index)
        model.addConstr(gen_P - Pd == branch_out - branch_in)
    
    model.optimize()
    return model, Pg, theta, P_branch, z


In [22]:
model, Pg, theta, P_branch, z = solve_dc_opf_full(bus_df, gen_df, branch_df, gencost_df, capacity_factor=0.9)

if model.status == GRB.OPTIMAL:
    print("=" * 70)
    print("IEEE 57-BUS DC-OPF WITH ALL CONSTRAINTS")
    print("Author: Jewook Park")
    print("=" * 70)
    print(f"Optimal Cost: ${model.objVal:.2f}")
    print(f"Total Generation: {sum(Pg[i].X for i in gen_df.index):.2f} MW")
    print(f"Total Load: {bus_df['Pd'].sum():.2f} MW")
    
    lines_on = sum(z[i].X > 0.5 for i in branch_df.index)
    print(f"\nLine Switching: {lines_on}/{len(branch_df)} lines ON")
    
    # Angle analysis
    angle_diffs = [abs(theta[branch_df.loc[idx, 'fbus']].X - theta[branch_df.loc[idx, 'tbus']].X) * 180 / np.pi 
                   for idx in branch_df.index if z[idx].X > 0.5]
    print(f"Max Angle Diff: {max(angle_diffs):.2f}° | Avg: {np.mean(angle_diffs):.2f}°")
    
    # Capacity utilization
    util = [(abs(P_branch[idx].X) / (branch_df.loc[idx, 'rateA'] * 0.9) * 100) 
            for idx in branch_df.index if branch_df.loc[idx, 'rateA'] > 0 and z[idx].X > 0.5]
    print(f"Max Capacity Util: {max(util):.1f}% | Avg: {np.mean(util):.1f}%")
    print("=" * 70)
else:
    print(f"Optimization failed: {model.status}")


Angle bounds: ±40.0° | Capacity factor: 0.9 | Big-M: 3234.00
Optimization failed: 3


In [23]:
# Test multiple combinations to find feasible configuration

print("Testing different angle bound and capacity factor combinations...\n")

# Test configurations (angle_degrees, capacity_factor)
configs = [
    (40, 1.0),   # Most relaxed capacity
    (45, 0.95),  
    (50, 0.92),
    (45, 1.0),   # Very relaxed
    (40, 0.95),  # Moderate
]

results = []

for angle_deg, cap_factor in configs:
    angle_rad = angle_deg * np.pi / 180
    
    # Create model
    model = gp.Model("DC-OPF-Test")
    model.Params.OutputFlag = 0
    ref_bus = bus_df[bus_df['type'] == 3]['bus_i'].values[0]
    
    max_susceptance = (1.0 / branch_df['x']).max()
    max_capacity = branch_df['rateA'].max()
    M = max(2 * angle_rad * max_susceptance, max_capacity * 2)
    
    # Variables
    Pg = model.addVars(gen_df.index, lb=0, name="Pg")
    theta = model.addVars(bus_df['bus_i'], lb=-angle_rad, ub=angle_rad, name="theta")
    P_branch = model.addVars(branch_df.index, lb=-GRB.INFINITY, name="P_branch")
    z = model.addVars(branch_df.index, vtype=GRB.BINARY, name="z")
    
    # Objective
    obj = gp.QuadExpr()
    for idx, row in gencost_df.iterrows():
        obj += row['c2'] * Pg[idx]*Pg[idx] + row['c1'] * Pg[idx] + row['c0']
    model.setObjective(obj, GRB.MINIMIZE)
    
    # Constraints
    model.addConstr(theta[ref_bus] == 0)
    
    for idx, row in gen_df.iterrows():
        model.addConstr(Pg[idx] >= row['Pmin'])
        model.addConstr(Pg[idx] <= row['Pmax'])
    
    for idx, row in branch_df.iterrows():
        fbus, tbus, x = row['fbus'], row['tbus'], row['x']
        B = 1.0 / x
        model.addConstr(P_branch[idx] - B * (theta[fbus] - theta[tbus]) <= M * (1 - z[idx]))
        model.addConstr(P_branch[idx] - B * (theta[fbus] - theta[tbus]) >= -M * (1 - z[idx]))
        model.addConstr(P_branch[idx] <= M * z[idx])
        model.addConstr(P_branch[idx] >= -M * z[idx])
    
    for idx, row in branch_df.iterrows():
        rateA = row['rateA'] * cap_factor
        if row['rateA'] > 0:
            model.addConstr(P_branch[idx] <= rateA * z[idx])
            model.addConstr(P_branch[idx] >= -rateA * z[idx])
    
    for idx, row in bus_df.iterrows():
        bus, Pd = row['bus_i'], row['Pd']
        gen_at_bus = gen_df[gen_df['bus'] == bus].index.tolist()
        gen_P = gp.quicksum(Pg[g] for g in gen_at_bus) if gen_at_bus else 0
        branch_out = gp.quicksum(P_branch[br] for br in branch_df[branch_df['fbus'] == bus].index)
        branch_in = gp.quicksum(P_branch[br] for br in branch_df[branch_df['tbus'] == bus].index)
        model.addConstr(gen_P - Pd == branch_out - branch_in)
    
    model.optimize()
    
    status_str = "✅ OPTIMAL" if model.status == GRB.OPTIMAL else f"❌ FAILED ({model.status})"
    cost = f"${model.objVal:.2f}" if model.status == GRB.OPTIMAL else "N/A"
    lines_on = sum(z[i].X > 0.5 for i in branch_df.index) if model.status == GRB.OPTIMAL else "N/A"
    
    print(f"Angle: ±{angle_deg}° | Capacity: {cap_factor:.2f} → {status_str} | Cost: {cost} | Lines: {lines_on}/80")
    
    results.append({
        'angle': angle_deg,
        'capacity': cap_factor,
        'status': model.status,
        'feasible': model.status == GRB.OPTIMAL
    })

print("\n" + "="*70)
feasible_configs = [r for r in results if r['feasible']]
if feasible_configs:
    print(f"✅ Found {len(feasible_configs)} feasible configuration(s)!")
    print("Tightest feasible: Angle ±{}°, Capacity {:.2f}".format(
        min(feasible_configs, key=lambda x: (x['capacity'], -x['angle']))['angle'],
        min(feasible_configs, key=lambda x: (x['capacity'], -x['angle']))['capacity']
    ))
else:
    print("❌ No feasible configurations found. Try more relaxed constraints.")
print("="*70)


Testing different angle bound and capacity factor combinations...

Angle: ±40° | Capacity: 1.00 → ❌ FAILED (3) | Cost: N/A | Lines: N/A/80
Angle: ±45° | Capacity: 0.95 → ❌ FAILED (3) | Cost: N/A | Lines: N/A/80
Angle: ±50° | Capacity: 0.92 → ❌ FAILED (3) | Cost: N/A | Lines: N/A/80
Angle: ±45° | Capacity: 1.00 → ❌ FAILED (3) | Cost: N/A | Lines: N/A/80
Angle: ±40° | Capacity: 0.95 → ❌ FAILED (3) | Cost: N/A | Lines: N/A/80

❌ No feasible configurations found. Try more relaxed constraints.


In [24]:
# Debug: Test without line switching (force all lines ON)

print("Testing WITHOUT line switching (all lines forced ON)...\n")

configs_debug = [
    (40, 1.0),
    (45, 1.0),
    (50, 1.0),
    (60, 1.0),
]

for angle_deg, cap_factor in configs_debug:
    angle_rad = angle_deg * np.pi / 180
    
    model = gp.Model("DC-OPF-NoSwitching")
    model.Params.OutputFlag = 0
    ref_bus = bus_df[bus_df['type'] == 3]['bus_i'].values[0]
    
    # Variables - NO BINARY z, all lines forced ON
    Pg = model.addVars(gen_df.index, lb=0, name="Pg")
    theta = model.addVars(bus_df['bus_i'], lb=-angle_rad, ub=angle_rad, name="theta")
    P_branch = model.addVars(branch_df.index, lb=-GRB.INFINITY, name="P_branch")
    
    # Objective
    obj = gp.QuadExpr()
    for idx, row in gencost_df.iterrows():
        obj += row['c2'] * Pg[idx]*Pg[idx] + row['c1'] * Pg[idx] + row['c0']
    model.setObjective(obj, GRB.MINIMIZE)
    
    # Constraints
    model.addConstr(theta[ref_bus] == 0)
    
    for idx, row in gen_df.iterrows():
        model.addConstr(Pg[idx] >= row['Pmin'])
        model.addConstr(Pg[idx] <= row['Pmax'])
    
    # DC power flow (NO Big-M, no switching)
    for idx, row in branch_df.iterrows():
        fbus, tbus, x = row['fbus'], row['tbus'], row['x']
        B = 1.0 / x
        model.addConstr(P_branch[idx] == B * (theta[fbus] - theta[tbus]))
    
    # Capacity constraints
    for idx, row in branch_df.iterrows():
        rateA = row['rateA'] * cap_factor
        if row['rateA'] > 0:
            model.addConstr(P_branch[idx] <= rateA)
            model.addConstr(P_branch[idx] >= -rateA)
    
    # Power balance
    for idx, row in bus_df.iterrows():
        bus, Pd = row['bus_i'], row['Pd']
        gen_at_bus = gen_df[gen_df['bus'] == bus].index.tolist()
        gen_P = gp.quicksum(Pg[g] for g in gen_at_bus) if gen_at_bus else 0
        branch_out = gp.quicksum(P_branch[br] for br in branch_df[branch_df['fbus'] == bus].index)
        branch_in = gp.quicksum(P_branch[br] for br in branch_df[branch_df['tbus'] == bus].index)
        model.addConstr(gen_P - Pd == branch_out - branch_in)
    
    model.optimize()
    
    status_str = "✅ OPTIMAL" if model.status == GRB.OPTIMAL else f"❌ FAILED ({model.status})"
    cost = f"${model.objVal:.2f}" if model.status == GRB.OPTIMAL else "N/A"
    
    print(f"Angle: ±{angle_deg}° | Capacity: {cap_factor:.2f} (NO SWITCHING) → {status_str} | Cost: {cost}")

print("\n💡 If these fail, the problem is the combination of angle bound + capacity cut.")
print("💡 If these work, the problem is adding line switching with Big-M.")


Testing WITHOUT line switching (all lines forced ON)...

Angle: ±40° | Capacity: 1.00 (NO SWITCHING) → ❌ FAILED (3) | Cost: N/A
Angle: ±45° | Capacity: 1.00 (NO SWITCHING) → ❌ FAILED (3) | Cost: N/A
Angle: ±50° | Capacity: 1.00 (NO SWITCHING) → ❌ FAILED (3) | Cost: N/A
Angle: ±60° | Capacity: 1.00 (NO SWITCHING) → ❌ FAILED (3) | Cost: N/A

💡 If these fail, the problem is the combination of angle bound + capacity cut.
💡 If these work, the problem is adding line switching with Big-M.
